# prelim-investigation

Preliminary investigation of LangChain chains and LLM setups. The models live in section 0; the chains go below it.

## 0. LLMs — this block is yours

Every model, key and URL lives here. A change of provider is a change in this block only.

Add the keys to `.env` at the project root:

```
LLM_API_KEY=...
TYPESAFE_API_KEY=...
BONSAI_API_KEY=not-needed
```

Model names and URLs have defaults, so `.env` only has to hold the keys. A missing key stops the cell with its name.

The cell finds the project folder itself, so the kernel working directory does not matter. If it cannot, set `PRELIM_ROOT` to the folder that holds `.env`.

In [3]:
# --------------------------------------------------------------- LLM setup
import os
from pathlib import Path

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_typesafe import TypeSafeClassifier

def _project_root():
    """The folder that holds .env and pyproject.toml, whatever the kernel cwd is."""
    starts = []
    notebook = globals().get("__vsc_ipynb_file__")   # set by VS Code
    if notebook:
        starts.append(Path(notebook).parent)
    starts += [Path.cwd(), Path.home() / "Codebase" / "prelim-investigation"]
    for start in starts:
        for folder in (start, *start.parents):
            if (folder / ".env").is_file() and (folder / "pyproject.toml").is_file():
                return folder
    raise FileNotFoundError("No folder with .env and pyproject.toml was found. "
                            "Start the kernel inside the project, or set PRELIM_ROOT.")


ROOT = Path(os.environ.get("PRELIM_ROOT") or _project_root())
ENV_FILE = ROOT / ".env"
load_dotenv(ENV_FILE)


def _env(name, default=None):
    value = os.environ.get(name, default)
    if not value:
        raise KeyError(f"{name} is not set: add it to {ENV_FILE}")
    return value


# The reasoning model. Breaks a question into sub-questions.
llm = ChatOpenAI(model_name=_env("LLM_MODEL", "deepseek-v4.1-flash"),
                 base_url=_env("LLM_BASE_URL", "https://opencode.ai/zen/v1"),
                 api_key=_env("LLM_API_KEY"),
                 temperature=2,
                 use_responses_api=False)

# The same model with thinking off. Calls that must be fast use this one.
quick_llm = ChatOpenAI(model_name=_env("QUICK_LLM_MODEL", "deepseek-v4.1-flash"),
                       base_url=_env("LLM_BASE_URL", "https://opencode.ai/zen/v1"),
                       api_key=_env("LLM_API_KEY"),
                       use_responses_api=False,
                       extra_body={"thinking": {"type": "disabled"}})

# A local model on port 6967.
bonsai_llm = ChatOpenAI(model=_env("BONSAI_MODEL", "bonsai"),
                        base_url=_env("BONSAI_BASE_URL", "http://localhost:6967/v1"),
                        api_key=_env("BONSAI_API_KEY", "not-needed"),
                        max_tokens=600)

# The router's classifier. It reads TYPESAFE_API_KEY from the environment.
jev = TypeSafeClassifier(model=_env("TYPESAFE_MODEL", "jev-1.13"),
                         base_url=_env("TYPESAFE_BASE_URL", "https://opencode.ai/zen"))

print("env:", ENV_FILE)
print("llm:", llm.model_name)
print("quick_llm:", quick_llm.model_name)
print("bonsai_llm:", bonsai_llm.model_name)
print("jev:", jev.model)


KeyError: 'LLM_API_KEY is not set: add it to .env'

## 1. The simple chain: points in, blog out

One prompt, one model call. The points are the only input, so the user writes
them and the chain does not change.

The instruction is a **system** message, as specified. `{points}` is filled when
the chain runs.

Nothing runs by itself: uncomment the call in the next cell when you are ready
to spend tokens.

In [ ]:
# ------------------------------------------------------- 1. The simple chain
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

blog_prompt = ChatPromptTemplate.from_messages([
    ("system", "Write a blog for publishing with the following points: {points}"),
])

simple_blog_chain = blog_prompt | llm | StrOutputParser()

In [ ]:
# Your input. Replace these points; any number of points is valid.
POINTS = """
- BrewContent is a new product for the future of Content creation.
- It has features for AI video, voice, image and text generation alongside automating the entire flow.
- Compared to competitors, which only service one of these features, there's no need to buy 4 different products when you can get one for the price of 5! 
"""

blog = simple_blog_chain.invoke({"points": POINTS})
print(blog)

## 2. The styled chain: examples, rules, then the points

Three inputs, one prompt, one model call:

| Input | What it holds |
|---|---|
| `examples` | Prose whose style the blog must match |
| `humanizing_rules` | Patterns the blog must avoid |
| `points` | The topics to write on; `POINTS` from section 1 |

The user supplies `examples` and `humanizing_rules` as text; the chain only
fills them into the prompt. Run section 1 first, because this chain reads
`POINTS` from there.

In [ ]:
# ------------------------------------------------------ 2. The styled chain
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

styled_prompt = ChatPromptTemplate.from_messages([
    ("system", "Write a blog for publishing that should be written in this prose style: "
               "{examples}, which should follow these rules in order to avoid llm smells: "
               "{humanizing_rules}, and which is written on the topic(s) provided in "
               "these points: {points}"),
])

styled_blog_chain = styled_prompt | llm | StrOutputParser()

In [ ]:
# Your input. Both blocks are free text; replace them as you like.
HUMANIZING_RULES = """don't use the following: “it’s not X, it’s Y” framings, rule-of-three lists, em-dash overuse, “delve/tapestry/landscape,” tidy summary endings, throat-clearing openers"""

EXAMPLES = """Two things get in the way of clear Founder identity and narrative:

1. The Fog of War: The first challenge is internal. Your story is still evolving, and you're too deep in the daily chaos to see its shape. You know something profound is happening in your company, but articulating it feels like trying to describe water while swimming. Besides, when the codebase has collapsed in a heap of on-call issues, identity and narrative are the last thing on your mind. You tell yourself you'll figure it out next week. Or the week after. You promise.

2. The Content-First Trap: The second challenge is external. Too many agencies and social media experts jump straight to content without getting context. They ask "What should we post?" before understanding why you're posting in the first place. They can tell you everything about your market position, but nothing about your authentic voice. You want to talk about your tech stack. They want you to pick a fight with Elon. End result? You sound like every other founder on LinkedIn."""

voice_blog = styled_blog_chain.invoke({
    "examples": EXAMPLES,
    "humanizing_rules": HUMANIZING_RULES,
    "points": POINTS,
})
print(voice_blog)